In [8]:
# Wczytanie datasetu Titanic
import pandas as pd
import numpy as np

# Wczytujemy dane Titanic
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

# Sprawdzamy strukturę danych
print(titanic.head())
print(f"\nWymiary: {titanic.shape}")
print(f"\nKolumny: {titanic.columns.tolist()}")

# PODSTAWOWE GRUPOWANIE
# Średni wiek według klasy podróży
sredni_wiek_klasa = titanic.groupby('Pclass')['Age'].mean()
print("\n=== Średni wiek według klasy ===")
print(sredni_wiek_klasa)

# Interpretacja:
# - Pierwsza klasa: pasażerowie starsi (wyższy status społeczny)
# - Trzecia klasa: pasażerowie młodsi (imigranci, robotnicy)

# Grupowanie według dwóch zmiennych: klasy i płci

# Obliczamy współczynnik przeżycia (survival rate)
survival_grouped = titanic.groupby(['Pclass', 'Sex'])['Survived'].agg([
  'count', # liczba pasażerów
  'sum', # liczba ocalonych
  'mean' # survival rate (%)
])

# Zmieniamy nazwy kolumn dla czytelności
survival_grouped.columns = ['Liczba_pasażerów', 'Liczba_ocalonych', 'Współczynnik_przeżycia']

# Formatujemy współczynnik jako procent
survival_grouped['Współczynnik_przeżycia_%'] = (survival_grouped['Współczynnik_przeżycia']
* 100).round(2)
print("=== Współczynnik przeżycia według klasy i płci ===")
print(survival_grouped)

# OBSERWACJE:
# - Kobiety miały znacznie wyższy survival rate niż mężczyźni ("kobiety i dzieci najpierw")
# - Pierwsza klasa miała lepsze szanse przeżycia
# - Najmniejsze szanse: mężczyźni z trzeciej klasy

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  

W

In [9]:
# WIELOFUNKCYJNE AGREGACJE
# Różne statystyki dla różnych kolumn jednocześnie
age_fare_stats = titanic.groupby('Pclass').agg({
'Age': ['mean', 'median', 'std', 'min', 'max'],
'Fare': ['mean', 'median', 'std'],
'Survived': ['sum', 'mean']
})
print("=== Statystyki wieku i opłaty według klasy ===")
print(age_fare_stats)

# Spłaszczamy MultiIndex kolumn dla wygody
age_fare_stats.columns = ['_'.join(col).strip() for col in age_fare_stats.columns.values]
age_fare_stats = age_fare_stats.reset_index()
print("\n=== Po spłaszczeniu ===")
print(age_fare_stats)


=== Statystyki wieku i opłaty według klasy ===
              Age                                     Fare           \
             mean median        std   min   max       mean   median   
Pclass                                                                
1       38.233441   37.0  14.802856  0.92  80.0  84.154687  60.2875   
2       29.877630   29.0  14.001077  0.67  70.0  20.662183  14.2500   
3       25.140620   24.0  12.495398  0.42  74.0  13.675550   8.0500   

                  Survived            
              std      sum      mean  
Pclass                                
1       78.380373      136  0.629630  
2       13.417399       87  0.472826  
3       11.778142      119  0.242363  

=== Po spłaszczeniu ===
   Pclass   Age_mean  Age_median    Age_std  Age_min  Age_max  Fare_mean  \
0       1  38.233441        37.0  14.802856     0.92     80.0  84.154687   
1       2  29.877630        29.0  14.001077     0.67     70.0  20.662183   
2       3  25.140620        24.0  12.49

In [10]:
# Przygotowanie tabeli przestawnej
survival_pivot = titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack() * 100
print("=== Współczynnik przeżycia (%) - tabela Pclass × Sex ===")
print(survival_pivot.round(2))

# Tekstowy "wykres słupkowy" - każdy █ = 2%
print("\nTekstowe porównanie (█ = 2%):")
for pclass, row in survival_pivot.iterrows():
  for sex in survival_pivot.columns:
    rate = row[sex]
    bar = '█' * int(rate // 2)
  print(f" Klasa {pclass}, {sex:6s} |{bar} {rate:5.1f}%")

# INTERPRETACJA:
# - Kobiety z 1. i 2. klasy: ~95% survival rate
# - Mężczyźni z 3. klasy: ~13% survival rate
# - Widoczna jest zarówno klasowa, jak i płciowa dysproporcja w szansach przeżycia


=== Współczynnik przeżycia (%) - tabela Pclass × Sex ===
Sex     female   male
Pclass               
1        96.81  36.89
2        92.11  15.74
3        50.00  13.54

Tekstowe porównanie (█ = 2%):
 Klasa 1, male   |██████████████████  36.9%
 Klasa 2, male   |███████  15.7%
 Klasa 3, male   |██████  13.5%


In [11]:
import pandas as pd
import numpy as np

# Wczytujemy dane o sprzedaży (symulowane)
np.random.seed(42)
sales_data = pd.DataFrame({
'Region': np.random.choice(['Północ', 'Południe', 'Wschód', 'Zachód'], 200),
'Produkt': np.random.choice(['A', 'B', 'C'], 200),
'Sprzedaż': np.random.randint(100, 1000, 200),
'Koszt': np.random.randint(50, 500, 200)
})
print(sales_data.head(10))

# AGGREGATION - zwraca jedną wartość na grupę
avg_sales_region = sales_data.groupby('Region')['Sprzedaż'].mean()
print("\n=== Średnia sprzedaż według regionu (Aggregation) ===")
print(avg_sales_region)
print(f"Rozmiar wyniku: {avg_sales_region.shape}") # (4,) - 4 regiony

# TRANSFORM - zwraca wartość dla każdego wiersza
sales_data['Średnia_region'] = sales_data.groupby('Region')['Sprzedaż'].transform('mean')
print("\n=== DataFrame z dodaną kolumną Transform ===")
print(sales_data.head(10))
print(f"Rozmiar wyniku: {sales_data.shape}") # (200, 5) - oryginalny rozmiar zachowany!
# Każdy wiersz ma przypisaną średnią swojego regionu
# To umożliwia porównanie konkretnej obserwacji z grupową średnią


     Region Produkt  Sprzedaż  Koszt
0    Wschód       C       596    317
1    Zachód       C       401    372
2    Północ       A       280    114
3    Wschód       A       706    466
4    Wschód       B       198    217
5    Zachód       A       799    379
6    Północ       C       215     92
7    Północ       C       290     93
8    Wschód       A       352    334
9  Południe       C       260    446

=== Średnia sprzedaż według regionu (Aggregation) ===
Region
Południe    548.760870
Północ      564.304348
Wschód      595.222222
Zachód      595.851852
Name: Sprzedaż, dtype: float64
Rozmiar wyniku: (4,)

=== DataFrame z dodaną kolumną Transform ===
     Region Produkt  Sprzedaż  Koszt  Średnia_region
0    Wschód       C       596    317      595.222222
1    Zachód       C       401    372      595.851852
2    Północ       A       280    114      564.304348
3    Wschód       A       706    466      595.222222
4    Wschód       B       198    217      595.222222
5    Zachód       A    

In [15]:
# STANDARYZACJA WEDŁUG GRUPY

# Normalizujemy sprzedaż wewnątrz każdego regionu (z-score)
def standaryzacja(x):
  """Standaryzacja: (x - mean) / std"""
  return (x - x.mean()) / x.std()

#Standaryzacja sprzedaży według regionu (poprawione przez usunięcie podziału linii)
sales_data['Sprzedaż_std_region'] = sales_data.groupby('Region')['Sprzedaż'].transform(standaryzacja)

# Sprawdzamy wyniki dla regionu 'Północ'
polnoc = sales_data[sales_data['Region'] == 'Północ'][['Region', 'Sprzedaż',
'Sprzedaż_std_region']].head(10)
print("\n=== Standaryzacja sprzedaży w regionie Północ ===")
print(polnoc)

# Weryfikacja: średnia standaryzowanych wartości w grupie powinna być ~0, std ~1
print("\n=== Weryfikacja standaryzacji ===")
verification = sales_data.groupby('Region')['Sprzedaż_std_region'].agg(['mean', 'std'])
print(verification.round(4))

# ZASTOSOWANIE:
# - Porównanie wyników w różnych skalach (różne regiony, różne produkty)
# - Wykrywanie outliers w kontekście grupy
# - Feature engineering dla modeli ML (poznamy modele ML w Lekcjach 20+)



=== Standaryzacja sprzedaży w regionie Północ ===
    Region  Sprzedaż  Sprzedaż_std_region
2   Północ       280            -1.078252
6   Północ       215            -1.324771
7   Północ       290            -1.040326
15  Północ       834             1.022848
21  Północ       891             1.239026
30  Północ       884             1.212477
31  Północ       203            -1.370283
35  Północ       275            -1.097215
37  Północ       576             0.044357
38  Północ       781             0.821840

=== Weryfikacja standaryzacji ===
          mean  std
Region             
Południe   0.0  1.0
Północ    -0.0  1.0
Wschód     0.0  1.0
Zachód     0.0  1.0


In [16]:
# Symulujemy dane z brakami
sales_missing = sales_data.copy()

# Losowo usuwamy 20% wartości w kolumnie Sprzedaż
np.random.seed(0)
missing_idx = np.random.choice(sales_missing.index, size=40, replace=False)
sales_missing.loc[missing_idx, 'Sprzedaż'] = np.nan
print(f"Liczba braków: {sales_missing['Sprzedaż'].isnull().sum()}")

# WYPEŁNIANIE MEDIANĄ GRUPOWĄ
# Zamiast globalnej mediany, używamy mediany z każdego regionu
sales_missing['Sprzedaż_filled'] = sales_missing.groupby('Region')['Sprzedaż'].transform(
lambda x: x.fillna(x.median())
)

# Porównanie: które wartości zostały wypełnione
filled_values = sales_missing[sales_missing['Sprzedaż'].isnull()][['Region', 'Sprzedaż',
'Sprzedaż_filled']].head(10)
print("\n=== Wypełnione wartości (braki zastąpione medianą regionu) ===")
print(filled_values)

# Sprawdzamy, czy braki zostały usunięte
print(f"\nLiczba braków po wypełnieniu: {sales_missing['Sprzedaż_filled'].isnull().sum()}")

Liczba braków: 40

=== Wypełnione wartości (braki zastąpione medianą regionu) ===
      Region  Sprzedaż  Sprzedaż_filled
4     Wschód       NaN            671.0
5     Zachód       NaN            605.0
7     Północ       NaN            570.0
12    Wschód       NaN            671.0
16    Zachód       NaN            605.0
18    Zachód       NaN            605.0
22  Południe       NaN            573.0
33  Południe       NaN            573.0
37    Północ       NaN            570.0
44    Zachód       NaN            605.0

Liczba braków po wypełnieniu: 0


In [18]:
import pandas as pd

# Przygotowanie: agregacja i transformacja na tych samych danych
print("=" * 60)
print("AGGREGATION: Średnia sprzedaż według regionu")
print("=" * 60)
avg_sales = sales_data.groupby('Region')['Sprzedaż'].mean().sort_values()
print(avg_sales.round(2))

# Tekstowy "wykres słupkowy" (█ = 50 jedn.)
print("\nTekstowe porównanie (█ ≈ 50 jedn.):")
for region, val in avg_sales.items():
  bar = '█' * int(val // 50)
  print(f" {region:10s} |{bar} {val:.1f}")

print("\n" + "=" * 60)
print("TRANSFORM: Wartość rzeczywista vs średnia regionu (próbka 10)")
print("=" * 60)

# Transform dodaje statystykę grupową jako nową kolumnę — porównujemy indywidualne
# obserwacje ze średnią regionu
sample_data = sales_data.sample(10, random_state=1).sort_values('Region')
sample_view = sample_data[['Region', 'Sprzedaż', 'Średnia_region']].copy()
sample_view['Różnica'] = sample_view['Sprzedaż'] - sample_view['Średnia_region']
print(sample_view.round(2).to_string(index=False))

AGGREGATION: Średnia sprzedaż według regionu
Region
Południe    548.76
Północ      564.30
Wschód      595.22
Zachód      595.85
Name: Sprzedaż, dtype: float64

Tekstowe porównanie (█ ≈ 50 jedn.):
 Południe   |██████████ 548.8
 Północ     |███████████ 564.3
 Wschód     |███████████ 595.2
 Zachód     |███████████ 595.9

TRANSFORM: Wartość rzeczywista vs średnia regionu (próbka 10)
  Region  Sprzedaż  Średnia_region  Różnica
Południe       345          548.76  -203.76
Południe       205          548.76  -343.76
Południe       966          548.76   417.24
  Północ       435          564.30  -129.30
  Północ       526          564.30   -38.30
  Wschód       242          595.22  -353.22
  Wschód       637          595.22    41.78
  Wschód       198          595.22  -397.22
  Wschód       621          595.22    25.78
  Zachód       485          595.85  -110.85


In [19]:
# Wracamy do datasetu Titanic
titanic_fe = titanic.copy()

# === FEATURE ENGINEERING ===

# 1. Rozmiar rodziny
titanic_fe['Family_Size'] = titanic_fe['SibSp'] + titanic_fe['Parch'] + 1
# SibSp = rodzeństwo/małżonkowie, Parch = rodzice/dzieci
# +1 bo liczymy też samego pasażera

# 2. Czy podróżował sam?
titanic_fe['Is_Alone'] = (titanic_fe['Family_Size'] == 1).astype(int)

# 3. Tytuł z imienia (Mr, Mrs, Miss, Master, etc.)
titanic_fe['Title'] = titanic_fe['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# 4. Grupowanie rzadkich tytułów
title_counts = titanic_fe['Title'].value_counts()
rare_titles = title_counts[title_counts < 10].index
titanic_fe['Title'] = titanic_fe['Title'].replace(rare_titles, 'Rare')

# 5. Cena biletu na osobę w rodzinie
titanic_fe['Fare_Per_Person'] = titanic_fe['Fare'] / titanic_fe['Family_Size']
print("=== Nowe cechy ===")
print(titanic_fe[['Name', 'Family_Size', 'Is_Alone', 'Title', 'Fare',
'Fare_Per_Person']].head(10))

# Analiza: Jak rozmiar rodziny wpływa na survival rate?
survival_by_family = titanic_fe.groupby('Family_Size')['Survived'].mean()
print("\n=== Survival rate według rozmiaru rodziny ===")
print(survival_by_family.round(3))
# OBSERWACJA:
# - Osoby samotne: niższy survival rate
# - Małe rodziny (2-4): wyższy survival rate
# - Bardzo duże rodziny (5+): niższy survival rate (trudniej ewakuować dużą grupę)


=== Nowe cechy ===
                                                Name  Family_Size  Is_Alone  \
0                            Braund, Mr. Owen Harris            2         0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...            2         0   
2                             Heikkinen, Miss. Laina            1         1   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)            2         0   
4                           Allen, Mr. William Henry            1         1   
5                                   Moran, Mr. James            1         1   
6                            McCarthy, Mr. Timothy J            1         1   
7                     Palsson, Master. Gosta Leonard            5         0   
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)            3         0   
9                Nasser, Mrs. Nicholas (Adele Achem)            2         0   

    Title     Fare  Fare_Per_Person  
0      Mr   7.2500          3.62500  
1     Mrs  71.2833         35.64165

In [20]:
# BINNING - podział wieku na kategorie

# Metoda 1: pd.cut() - równe przedziały
titanic_fe['Age_Bin_Equal'] = pd.cut(
titanic_fe['Age'],
bins=5, # 5 równych przedziałów
labels=['Bardzo młody', 'Młody', 'Średni', 'Starszy', 'Bardzo starszy']
)

# Metoda 2: pd.cut() - własne przedziały
titanic_fe['Age_Group'] = pd.cut(
titanic_fe['Age'],
bins=[0, 12, 18, 35, 60, 100],
labels=['Dziecko', 'Nastolatek', 'Młody dorosły', 'Dorosły', 'Senior']
)

# Metoda 3: pd.qcut() - kwantyle (równa liczba obserwacji w każdym przedziale)
titanic_fe['Age_Quantile'] = pd.qcut(
titanic_fe['Age'],
q=4, # quartiles
labels=['Q1', 'Q2', 'Q3', 'Q4']
)
print("=== Porównanie metod binowania ===")
print(titanic_fe[['Age', 'Age_Bin_Equal', 'Age_Group', 'Age_Quantile']].head(20))

# Analiza survival rate według grup wiekowych
survival_age = titanic_fe.groupby('Age_Group', observed=False)['Survived'].agg(['count',
'mean'])
survival_age.columns = ['Liczba', 'Survival_Rate']
print("\n=== Survival rate według grup wiekowych ===")
print(survival_age.round(3))


=== Porównanie metod binowania ===
     Age Age_Bin_Equal      Age_Group Age_Quantile
0   22.0         Młody  Młody dorosły           Q2
1   38.0        Średni        Dorosły           Q3
2   26.0         Młody  Młody dorosły           Q2
3   35.0        Średni  Młody dorosły           Q3
4   35.0        Średni  Młody dorosły           Q3
5    NaN           NaN            NaN          NaN
6   54.0       Starszy        Dorosły           Q4
7    2.0  Bardzo młody        Dziecko           Q1
8   27.0         Młody  Młody dorosły           Q2
9   14.0  Bardzo młody     Nastolatek           Q1
10   4.0  Bardzo młody        Dziecko           Q1
11  58.0       Starszy        Dorosły           Q4
12  20.0         Młody  Młody dorosły           Q1
13  39.0        Średni        Dorosły           Q4
14  14.0  Bardzo młody     Nastolatek           Q1
15  55.0       Starszy        Dorosły           Q4
16   2.0  Bardzo młody        Dziecko           Q1
17   NaN           NaN            NaN          

In [22]:
# Tworzymy dataset ze sprzedażą czasową
np.random.seed(42)
date_range = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')

sales_time = pd.DataFrame({
    'Data': date_range,
    'Sprzedaż': np.random.poisson(lam=150, size=len(date_range)) +
np.sin(np.arange(len(date_range)) / 30) * 30
})

# === EKSTRAKCJA CECH Z DATY ===
sales_time['Rok'] = sales_time['Data'].dt.year
sales_time['Miesiąc'] = sales_time['Data'].dt.month
sales_time['Dzień'] = sales_time['Data'].dt.day
sales_time['Dzień_tygodnia'] = sales_time['Data'].dt.dayofweek # 0=Poniedziałek, 6=Niedziela
sales_time['Nazwa_dnia'] = sales_time['Data'].dt.day_name()
sales_time['Kwartał'] = sales_time['Data'].dt.quarter
sales_time['Czy_weekend'] = (sales_time['Dzień_tygodnia'] >= 5).astype(int)
sales_time['Tydzień_roku'] = sales_time['Data'].dt.isocalendar().week

# Sezon (meteorologiczny)
def get_season(month):
    if month in [12, 1, 2]:
        return 'Zima'
    elif month in [3, 4, 5]:
        return 'Wiosna'
    elif month in [6, 7, 8]:
        return 'Lato'
    else:
        return 'Jesień'

sales_time['Sezon'] = sales_time['Miesiąc'].apply(get_season)

print("=== Feature engineering z daty ===")
print(sales_time.head(15))

# Analiza: Średnia sprzedaż według dnia tygodnia
avg_sales_day = sales_time.groupby('Nazwa_dnia')['Sprzedaż'].mean().reindex([
'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])

print("\n=== Średnia sprzedaż według dnia tygodnia ===")
print(avg_sales_day.round(2))


=== Feature engineering z daty ===
         Data    Sprzedaż   Rok  Miesiąc  Dzień  Dzień_tygodnia Nazwa_dnia  \
0  2023-01-01  145.000000  2023        1      1               6     Sunday   
1  2023-01-02  159.999815  2023        1      2               0     Monday   
2  2023-01-03  137.998519  2023        1      3               1    Tuesday   
3  2023-01-04  156.995002  2023        1      4               2  Wednesday   
4  2023-01-05  166.988159  2023        1      5               3   Thursday   
5  2023-01-06  141.976884  2023        1      6               4     Friday   
6  2023-01-07  148.960080  2023        1      7               5   Saturday   
7  2023-01-08  154.936654  2023        1      8               6     Sunday   
8  2023-01-09  161.905522  2023        1      9               0     Monday   
9  2023-01-10  150.865606  2023        1     10               1    Tuesday   
10 2023-01-11  157.815841  2023        1     11               2  Wednesday   
11 2023-01-12  148.755170  20

In [23]:
# Analiza: Jak nowe cechy wpływają na survival rate — 4 widoki w tabelach

print("=" * 60)
print("1. Survival Rate według Rozmiaru Rodziny")
print("=" * 60)
survival_family = (titanic_fe.groupby('Family_Size')['Survived'].mean() * 100).round(1)
for size, rate in survival_family.items():
  bar = '█' * int(rate // 2)
  print(f" Family_Size = {size:2d} |{bar} {rate:.1f}%")

print("\n" + "=" * 60)
print("2. Survival Rate: Sam vs Nie Sam")
print("=" * 60)
survival_alone = (titanic_fe.groupby('Is_Alone')['Survived'].mean() * 100).round(1)
labels_alone = {0: 'Nie sam', 1: 'Sam'}
for flag, rate in survival_alone.items():
  bar = '█' * int(rate // 2)
  print(f" {labels_alone[flag]:8s} |{bar} {rate:.1f}%")

print("\n" + "=" * 60)
print("3. Survival Rate według Tytułu")
print("=" * 60)
survival_title = (titanic_fe.groupby('Title')['Survived'].mean() *
100).round(1).sort_values()
for title, rate in survival_title.items():
  bar = '█' * int(rate // 2)
  print(f" {title:10s} |{bar} {rate:.1f}%")

print("\n" + "=" * 60)
print("4. Survival Rate według Grupy Wiekowej")
print("=" * 60)
survival_age_group = (titanic_fe.groupby('Age_Group', observed=False)['Survived'].mean() *
100).round(1)
for group, rate in survival_age_group.items():
  bar = '█' * int(rate // 2)
  print(f" {str(group):15s} |{bar} {rate:.1f}%")
# WNIOSKI:
# - Tytuł "Mrs" (zamężne kobiety): wysoki survival rate
# - Dzieci: wysoki survival rate ("dzieci najpierw")
# - Małe rodziny (2-4): lepsze szanse niż osoby samotne lub bardzo duże rodziny

1. Survival Rate według Rozmiaru Rodziny
 Family_Size =  1 |███████████████ 30.4%
 Family_Size =  2 |███████████████████████████ 55.3%
 Family_Size =  3 |████████████████████████████ 57.8%
 Family_Size =  4 |████████████████████████████████████ 72.4%
 Family_Size =  5 |██████████ 20.0%
 Family_Size =  6 |██████ 13.6%
 Family_Size =  7 |████████████████ 33.3%
 Family_Size =  8 | 0.0%
 Family_Size = 11 | 0.0%

2. Survival Rate: Sam vs Nie Sam
 Nie sam  |█████████████████████████ 50.6%
 Sam      |███████████████ 30.4%

3. Survival Rate według Tytułu
 Mr         |███████ 15.7%
 Rare       |██████████████████████ 44.4%
 Master     |████████████████████████████ 57.5%
 Miss       |██████████████████████████████████ 69.8%
 Mrs        |███████████████████████████████████████ 79.2%

4. Survival Rate według Grupy Wiekowej
 Dziecko         |█████████████████████████████ 58.0%
 Nastolatek      |█████████████████████ 42.9%
 Młody dorosły   |███████████████████ 38.3%
 Dorosły         |███████████████

In [25]:
# WŁASNE FUNKCJE AGREGUJĄCE

# Funkcja obliczająca zakres (różnica max - min)
def range_func(x):
  return x.max() - x.min()

# Funkcja obliczająca współczynnik zmienności (CV)
def coef_variation(x):
  return (x.std() / x.mean()) * 100 if x.mean() != 0 else 0

# Funkcja zliczająca outliers (wartości > 2 std od średniej)
def count_outliers(x):
  mean = x.mean()
  std = x.std()
  outliers = ((x < mean - 2*std) | (x > mean + 2*std)).sum()
  return outliers

# Zastosowanie custom funkcji
custom_stats = sales_data.groupby('Region')['Sprzedaż'].agg([
  'mean',
  'std',
  ('zakres', range_func),
  ('CV_%', coef_variation),
  ('outliers_count', count_outliers)
]).round(2)

print("=== Zaawansowane statystyki według regionu ===")
print(custom_stats)

# INTERPRETACJA:
# - CV% (Coefficient of Variation): im wyższy, tym większa zmienność względna
# - Zakres: różnica między max a min sprzedażą
# - Outliers: liczba wartości odstających (>2 std)


=== Zaawansowane statystyki według regionu ===
            mean     std  zakres   CV_%  outliers_count
Region                                                 
Południe  548.76  275.67     868  50.24               0
Północ    564.30  263.67     892  46.73               0
Wschód    595.22  279.46     878  46.95               0
Zachód    595.85  265.04     860  44.48               0


In [26]:
# FILTER - wybór tylko tych GRUP, które spełniają warunek

# Wybieramy tylko regiony, w których średnia sprzedaż > 500
high_sales_regions = sales_data.groupby('Region').filter(lambda x: x['Sprzedaż'].mean() >
500)
print(f"Dane przed filtrowaniem grup: {sales_data.shape[0]} wierszy")
print(f"Dane po filtrowaniu grup: {high_sales_regions.shape[0]} wierszy")
print("\n=== Średnia sprzedaż w pozostałych regionach ===")
print(high_sales_regions.groupby('Region')['Sprzedaż'].mean().round(2))

# Filter vs Boolean Indexing:
# - Filter usuwa CAŁE GRUPY
# - Boolean indexing usuwa pojedyncze WIERSZE

# Przykład 2: Wybieramy tylko grupy z min. 40 obserwacjami
large_groups = sales_data.groupby('Region').filter(lambda x: len(x) >= 40)
print("\n=== Liczba obserwacji według regionu (tylko duże grupy) ===")
print(large_groups.groupby('Region').size())

Dane przed filtrowaniem grup: 200 wierszy
Dane po filtrowaniu grup: 200 wierszy

=== Średnia sprzedaż w pozostałych regionach ===
Region
Południe    548.76
Północ      564.30
Wschód      595.22
Zachód      595.85
Name: Sprzedaż, dtype: float64

=== Liczba obserwacji według regionu (tylko duże grupy) ===
Region
Południe    46
Północ      46
Wschód      54
Zachód      54
dtype: int64


In [28]:
# APPLY - elastyczna funkcja, która może zwracać Series, DataFrame lub skalar

# Funkcja normalizująca wartości w grupie do zakresu [0, 1]
def min_max_normalize(group):
  min_val = group['Sprzedaż'].min()
  max_val = group['Sprzedaż'].max()
  group = group.copy()
  group['Sprzedaż_norm'] = (group['Sprzedaż'] - min_val) / (max_val - min_val)
  return group

# Normalizacja wewnątrz każdego regionu
sales_normalized = sales_data.groupby('Region', group_keys=False).apply(min_max_normalize)
print("=== Dane z normalizacją grupową ===")
print(sales_normalized.head(20))

# Weryfikacja: w każdym regionie min=0, max=1
verification = sales_normalized.groupby('Region')['Sprzedaż_norm'].agg(['min', 'max'])
print("\n=== Weryfikacja normalizacji ===")
print(verification)

# ZASTOSOWANIE:
# - Porównanie wyników na różnych skalach
# - Feature scaling w kontekście grupowym (użyjemy w ML w Lekcjach 20+)
# - Identyfikacja najlepszych/najgorszych w każdej grupie

=== Dane z normalizacją grupową ===
      Region Produkt  Sprzedaż  Koszt  Średnia_region  Sprzedaż_std_region  \
0     Wschód       C       596    317      595.222222             0.002783   
1     Zachód       C       401    372      595.851852            -0.735182   
2     Północ       A       280    114      564.304348            -1.078252   
3     Wschód       A       706    466      595.222222             0.396406   
4     Wschód       B       198    217      595.222222            -1.421415   
5     Zachód       A       799    379      595.851852             0.766485   
6     Północ       C       215     92      564.304348            -1.324771   
7     Północ       C       290     93      564.304348            -1.040326   
8     Wschód       A       352    334      595.222222            -0.870343   
9   Południe       C       260    446      548.760870            -1.047482   
10    Wschód       C       355     61      595.222222            -0.859608   
11    Wschód       A       4

/tmp/ipykernel_827/2967821348.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sales_normalized = sales_data.groupby('Region', group_keys=False).apply(min_max_normalize)


In [29]:
# WYKRYWANIE OUTLIERS W KONTEKŚCIE GRUPY
def detect_outliers_iqr(group):
  """Wykrywa outliery metodą IQR (Interquartile Range)"""
  Q1 = group['Sprzedaż'].quantile(0.25)
  Q3 = group['Sprzedaż'].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  group = group.copy()
  group['Is_Outlier'] = ((group['Sprzedaż'] < lower_bound) | (group['Sprzedaż'] >
  upper_bound)).astype(int)
  return group

# Wykrywamy outliery w każdym regionie osobno
sales_outliers = sales_data.groupby('Region', group_keys=False).apply(detect_outliers_iqr)

# Liczba outliers według regionu
outliers_count = sales_outliers.groupby('Region')['Is_Outlier'].sum()
print("=== Liczba outliers według regionu (metoda IQR) ===")
print(outliers_count)

# Pokazujemy przykładowe outliery
print("\n=== Przykładowe outliery ===")
print(sales_outliers[sales_outliers['Is_Outlier'] == 1].head(10))

# ZASTOSOWANIE:
# - Wykrywanie anomalii w kontekście lokalnym (grupa)
# - Wartość może być outlierem w jednym regionie, ale normalna w innym


=== Liczba outliers według regionu (metoda IQR) ===
Region
Południe    0
Północ      0
Wschód      0
Zachód      0
Name: Is_Outlier, dtype: int64

=== Przykładowe outliery ===
Empty DataFrame
Columns: [Region, Produkt, Sprzedaż, Koszt, Średnia_region, Sprzedaż_std_region, Is_Outlier]
Index: []


/tmp/ipykernel_827/1583906893.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sales_outliers = sales_data.groupby('Region', group_keys=False).apply(detect_outliers_iqr)


In [30]:
# Tekstowa reprezentacja boxplot: podajemy 5 statystyk (min/Q1/mediana/Q3/max)
# oraz wizualizację w stylu `min---|Q1===median===Q3|---max` dla każdego regionu.

print("=== Rozkład sprzedaży według regionu (tekstowy boxplot) ===\n")

stats = sales_data.groupby('Region')['Sprzedaż'].agg([
  'min',
  lambda x: x.quantile(0.25),
  'median',
  lambda x: x.quantile(0.75),
  'max'
])
stats.columns = ['min', 'Q1', 'med', 'Q3', 'max']
print(stats.round(1))

# Rysujemy prostą tekstową skalę
overall_min = sales_data['Sprzedaż'].min()
overall_max = sales_data['Sprzedaż'].max()
width = 50 # szerokość tekstowej skali

def pos(val):
  """Zwraca pozycję (0..width-1) dla danej wartości na skali"""
  return int((val - overall_min) / (overall_max - overall_min) * (width - 1))

print("\nTekstowy boxplot (| = kwartyle, - = zakres, * = mediana):")
for region, row in stats.iterrows():
  line = [' '] * width
  # zakres min..max = "-"
  for i in range(pos(row['min']), pos(row['max']) + 1):
    line[i] = '-'
  # Q1..Q3 = "="
  for i in range(pos(row['Q1']), pos(row['Q3']) + 1):
    line[i] = '='
  # mediana = "*"
  line[pos(row['med'])] = '*'
  # krawędzie
  line[pos(row['min'])] = '|'
  line[pos(row['max'])] = '|'
  print(f" {region:10s} |{''.join(line)}|")

# INTERPRETACJA:
# - Różne regiony mają różne rozkłady sprzedaży
# - Szerszy zakres `=` = większa zmienność (IQR)
# - Pozycja `*` = mediana
# - W Lekcji 17 zobaczymy wersję graficzną (matplotlib.boxplot + seaborn.violinplot)

=== Rozkład sprzedaży według regionu (tekstowy boxplot) ===

          min     Q1    med     Q3  max
Region                                 
Południe  115  267.2  592.5  761.0  983
Północ    104  338.5  549.0  801.5  996
Wschód    117  359.0  629.0  856.0  995
Zachód    122  387.2  617.5  797.0  982

Tekstowy boxplot (| = kwartyle, - = zakres, * = mediana):
 Południe   ||-------==================*==========-----------| |
 Północ     ||-----------============*==============----------||
 Wschód     ||-------------==============*=============------| |
 Zachód     ||--------------=============*==========---------| |


In [34]:
# === GREEN IT W PANDAS GROUPBY ===
# 1. USUWANIE DUPLIKATÓW PRZED AGREGACJĄ
# Duplikaty zwiększają zużycie pamięci i czas obliczeń

# Sprawdzamy duplikaty
duplicates_count = sales_data.duplicated().sum()
print(f"Liczba duplikatów: {duplicates_count}")

# Usuwamy duplikaty
sales_clean = sales_data.drop_duplicates()
print(f"Rozmiar przed: {sales_data.shape[0]}, po: {sales_clean.shape[0]}")

# 2. AGREGACJA ZAMIAST PEŁNYCH DANYCH
# Przechowuj zagregowane statystyki zamiast surowych danych (gdy możliwe)

# Zamiast przechowywać 200 wierszy:
print(f"Pamięć pełnego datasetu: {sales_data.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Przechowuj agregaty (4 wiersze):
sales_aggregated = sales_data.groupby("Region").agg({
  "Sprzedaż": ["mean", "sum", "count"],
  "Koszt": ["mean", "sum"]
})
print(f"Pamięć zagregowanych danych: {sales_aggregated.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Oszczędność pamięci:
savings = (1 - sales_aggregated.memory_usage(deep=True).sum() / sales_data.memory_usage(deep=True).sum()) * 100
print(f"Oszczędność pamięci: {savings:.1f}%")

# 3. OPTYMALIZACJA TYPÓW DANYCH PO AGREGACJI
# Po agregacji możemy zoptymalizować typy

sales_agg_optimized = sales_aggregated.copy()
# Konwersja float64 -> float32 (wystarczająca precyzja, mniej pamięci)
for col in sales_agg_optimized.select_dtypes(include=["float64"]).columns:
  sales_agg_optimized[col] = sales_agg_optimized[col].astype("float32")

# Poprawiono multiline f-string
print(f"\nPamięć przed optymalizacją typów: {sales_aggregated.memory_usage(deep=True).sum()} bytes")
print(f"Pamięć po optymalizacji typów: {sales_agg_optimized.memory_usage(deep=True).sum()} bytes")

# 4. GRUPOWANIE RZADKICH KATEGORII
# Zmniejsza liczbę unikalnych wartości = mniej pamięci i szybsze operacje

# Przykład: grupujemy rzadkie tytuły w Titanic
title_counts = titanic_fe["Title"].value_counts()
print("\n=== Liczba tytułów przed grupowaniem ===")
print(f"Unikalne tytuły: {titanic_fe['Title'].nunique()}")
print(title_counts)

# Grupujemy rzadkie (< 10 wystąpień) - już zrobiliśmy to wcześniej; dla porządku pokażemy
# dtypes  <- zakomentowano niepoprawną linię
titanic_fe["Title_Category"] = titanic_fe["Title"].astype("category")
mem_object = titanic_fe["Title"].memory_usage(deep=True)
mem_category = titanic_fe["Title_Category"].memory_usage(deep=True)
print(f"\nPamięć (object): {mem_object} bytes")
print(f"Pamięć (category): {mem_category} bytes")
print(f"Oszczędność: {(1 - mem_category/mem_object)*100:.1f}%")

Liczba duplikatów: 0
Rozmiar przed: 200, po: 200
Pamięć pełnego datasetu: 32.52 KB
Pamięć zagregowanych danych: 0.49 KB
Oszczędność pamięci: 98.5%

Pamięć przed optymalizacją typów: 497 bytes
Pamięć po optymalizacji typów: 465 bytes

=== Liczba tytułów przed grupowaniem ===
Unikalne tytuły: 5
Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64

Pamięć (object): 46276 bytes
Pamięć (category): 1459 bytes
Oszczędność: 96.8%


In [35]:
# ✅ DOBRE - próbka podczas testowania
sample_df = sales_data.sample(50, random_state=42)
result_test = sample_df.groupby('Region').apply(lambda x: x.head(2))
print("Test na próbce zakończony - kod działa!")

Test na próbce zakończony - kod działa!


/tmp/ipykernel_827/382192302.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_test = sample_df.groupby('Region').apply(lambda x: x.head(2))


In [36]:
import time
import numpy as np
import pandas as pd

# Zbuduj większy zbiór z losowych sprzedaży w 4 regionach
rng = np.random.default_rng(42)
n = 50_000
big_sales = pd.DataFrame({
  'Region': rng.choice(['Pn', 'Pd', 'Wsch', 'Zach'], size=n),
  'Sprzedaż': rng.normal(1000, 250, size=n),
})

# Wariant 1: .apply() — wolniejszy (funkcja Pythona per grupa)
durations_apply = []
for _ in range(3):
  start = time.perf_counter()
  wynik_apply = big_sales.groupby('Region').apply(
    lambda x: (x['Sprzedaż'] - x['Sprzedaż'].mean()) / x['Sprzedaż'].std()
  )
  durations_apply.append(time.perf_counter() - start)

# Wariant 2: .transform() — szybszy (ścieżka wektorowa w C)
durations_transform = []
for _ in range(3):
  start = time.perf_counter()
  wynik_transform = big_sales.groupby('Region')['Sprzedaż'].transform(
    lambda x: (x - x.mean()) / x.std()
  )
  durations_transform.append(time.perf_counter() - start)

t_apply = sum(durations_apply) / len(durations_apply)
t_transform = sum(durations_transform) / len(durations_transform)

print(f".apply() : {t_apply:.4f} s")
print(f".transform() : {t_transform:.4f} s")
print(f"Przyspieszenie: {t_apply / t_transform:.2f}×")

# Sprawdzamy równość liczbową:
# - .apply() tworzy MultiIndex (Region, oryginalny_index) → usuwamy poziom Region przez droplevel(0)
# - .transform() zachowuje oryginalny indeks DataFrame
wynik_apply_flat = wynik_apply.droplevel(0)
print("\nCzy wyniki są liczbowo równe?",
  np.allclose(wynik_apply_flat.sort_index().values,
    wynik_transform.sort_index().values))

.apply() : 0.0134 s
.transform() : 0.0085 s
Przyspieszenie: 1.58×

Czy wyniki są liczbowo równe? True


/tmp/ipykernel_827/3956206456.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wynik_apply = big_sales.groupby('Region').apply(
/tmp/ipykernel_827/3956206456.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wynik_apply = big_sales.groupby('Region').apply(
/tmp/ipykernel_827/3956206456.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and

In [38]:
# === FEATURE ENGINEERING DLA ML ===
# Dataset Titanic - przygotowanie do predykcji survival
ml_data = titanic_fe[['Pclass', 'Sex', 'Age', 'Family_Size', 'Fare_Per_Person',
'Is_Alone', 'Title', 'Survived']].copy()

# 1. Wypełniamy braki medianą grupową (według klasy i płci)
ml_data['Age'] = ml_data.groupby(['Pclass', 'Sex'])['Age'].transform(
  lambda x: x.fillna(x.median())
)

# 2. Kodowanie zmiennych kategorycznych — natywnie w Pandas
# Binary mapping (dla 2 klas):
ml_data['Sex_Encoded'] = ml_data['Sex'].map({'male': 0, 'female': 1})

# Factorize (dla wielu klas - zwraca kody 0,1,2,...)
codes, uniques = pd.factorize(ml_data['Title'])
ml_data['Title_Encoded'] = codes
print("Kodowanie tytułów (mapa):")
for code, uniq in enumerate(uniques):
  print(f" {code} -> {uniq}")

# 3. Agregacje grupowe jako cechy (target encoding)
ml_data['Avg_Survival_Pclass'] = ml_data.groupby('Pclass')['Survived'].transform('mean')
ml_data['Avg_Survival_Title'] = ml_data.groupby('Title')['Survived'].transform('mean')

# 4. Standaryzacja cech numerycznych grupowo
for col in ['Age', 'Fare_Per_Person']:
  ml_data[f'{col}_Std'] = ml_data.groupby('Pclass')[col].transform(
    lambda x: (x - x.mean()) / x.std()
  )

print("=== Dataset przygotowany do ML ===")
print(ml_data.head(10))
print(f"\nShape: {ml_data.shape}")
print(f"Braki: {ml_data.isnull().sum().sum()}")

# Dataset jest gotowy — kolejny krok (trenowanie modelu) poznamy w Lekcjach 20+:
# # zapowiedź Lekcji 22+ (scikit-learn nie jest jeszcze poznany):
# # from sklearn.ensemble import RandomForestClassifier
# # X = ml_data.drop(['Survived', 'Sex', 'Title'], axis=1)
# # y = ml_data['Survived']
# # model = RandomForestClassifier().fit(X, y)


Kodowanie tytułów (mapa):
 0 -> Mr
 1 -> Mrs
 2 -> Miss
 3 -> Master
 4 -> Rare
=== Dataset przygotowany do ML ===
   Pclass     Sex   Age  Family_Size  Fare_Per_Person  Is_Alone   Title  \
0       3    male  22.0            2          3.62500         0      Mr   
1       1  female  38.0            2         35.64165         0     Mrs   
2       3  female  26.0            1          7.92500         1    Miss   
3       1  female  35.0            2         26.55000         0     Mrs   
4       3    male  35.0            1          8.05000         1      Mr   
5       3    male  25.0            1          8.45830         1      Mr   
6       1    male  54.0            1         51.86250         1      Mr   
7       3    male   2.0            5          4.21500         0  Master   
8       3  female  27.0            3          3.71110         0     Mrs   
9       2  female  14.0            2         15.03540         0     Mrs   

   Survived  Sex_Encoded  Title_Encoded  Avg_Survival_Pclas

In [40]:
import pandas as pd
import numpy as np

# === GENEROWANIE SYNTETYCZNEGO DATASETU CALIFORNIA HOUSING ===
def make_california_housing(n=5000, seed=42):
  """Generuje syntetyczny dataset wzorowany na California Housing."""
  rng = np.random.default_rng(seed)

  # Współrzędne geograficzne w obszarze Kalifornii (szerokość ~32-42, długość ~-124..-114)
  latitude = rng.uniform(32.5, 41.5, n)
  longitude = rng.uniform(-123.5, -114.5, n)
  # Dochód: rozkład skośny
  med_inc = rng.gamma(shape=2.0, scale=1.8, size=n) + 0.5
  # Wiek domu
  house_age = rng.integers(1, 55, n)
  # Pokoje/sypialnie/populacja
  ave_rooms = rng.normal(5.5, 1.5, n).clip(1, 15)
  ave_bedrms = (ave_rooms * rng.uniform(0.15, 0.25, n)).clip(0.5, 5)
  population = rng.integers(50, 5000, n)
  ave_occup = rng.normal(3.0, 0.8, n).clip(1, 10)
  # Cena domu (w 100_000 USD) - zależna od dochodu, wieku i lokalizacji
  med_house_val = (
    med_inc * 0.6
    + (latitude - 36) * 0.15
    - (house_age / 100)
    + rng.normal(0, 0.3, n)
  ).clip(0.15, 5.0)

  return pd.DataFrame({
    'MedInc': med_inc.round(4),
    'HouseAge': house_age,
    'AveRooms': ave_rooms.round(4),
    'AveBedrms': ave_bedrms.round(4),
    'Population': population,
    'AveOccup': ave_occup.round(4),
    'Latitude': latitude.round(4),
    'Longitude': longitude.round(4),
    'MedHouseVal': med_house_val.round(4)
  })

df_housing = make_california_housing(n=5000, seed=42)

print("=== Syntetyczny California Housing Dataset ===")
print(df_housing.head())
print(f"\nShape: {df_housing.shape}")
print(f"\nOpis:\n{df_housing.describe().round(3)}")


=== Syntetyczny California Housing Dataset ===
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  0.9067        48    4.1561     0.9944        3929    2.5674   39.4656   
1  7.1731        51    6.9524     1.3357        3825    4.9413   36.4499   
2  4.8817        22    5.0764     0.9349        3550    3.2407   40.2274   
3  4.0980        54    3.3280     0.8064        3875    2.5277   38.7763   
4  5.1915         5    6.3899     1.1873        4516    3.7003   33.3476   

   Longitude  MedHouseVal  
0  -121.5367       0.4077  
1  -117.5337       4.0278  
2  -116.5982       3.2236  
3  -122.0003       3.0991  
4  -123.1053       2.2712  

Shape: (5000, 9)

Opis:
         MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  \
count  5000.000  5000.000  5000.000   5000.000    5000.000  5000.000   
mean      4.126    27.942     5.564      1.114    2489.534     2.991   
std       2.610    15.671     1.505      0.335    1427.849     0.803   
min       0.531

In [41]:
# === EKSPLORACJA ===
# 1. Tworzymy kategorie geograficzne (regiony) na podstawie Latitude/Longitude
# Północ/Południe: Latitude > 36
# Wschód/Zachód: Longitude > -119
df_housing['Region'] = 'Unknown'
df_housing.loc[(df_housing['Latitude'] >= 36) & (df_housing['Longitude'] >= -119),
'Region'] = 'NE'
df_housing.loc[(df_housing['Latitude'] >= 36) & (df_housing['Longitude'] < -119),
'Region'] = 'NW'
df_housing.loc[(df_housing['Latitude'] < 36) & (df_housing['Longitude'] >= -119),
'Region'] = 'SE'
df_housing.loc[(df_housing['Latitude'] < 36) & (df_housing['Longitude'] < -119),
'Region'] = 'SW'

print("=== Liczba obserwacji według regionu ===")
print(df_housing['Region'].value_counts())

# 2. Kategorie dochodów
df_housing['Income_Category'] = pd.cut(
  df_housing['MedInc'],
  bins=[0, 2.5, 4.5, 6, 15],
  labels=['Low', 'Medium', 'High', 'Very High']
)

# 3. Kategorie wieku domu
df_housing['Age_Category'] = pd.cut(
df_housing['HouseAge'],
bins=[0, 10, 25, 40, 60],
labels=['New', 'Modern', 'Old', 'Very Old']
)

# === AGREGACJE GRUPOWE ===
# Statystyki według regionu
regional_stats = df_housing.groupby('Region').agg({
  'MedHouseVal': ['mean', 'median', 'std'],
  'MedInc': ['mean', 'median'],
  'Population': ['mean', 'sum'],
  'HouseAge': 'mean'
}).round(2)

print("\n=== Statystyki według regionu ===")
print(regional_stats)

# Statystyki według kategorii dochodów
income_stats = df_housing.groupby('Income_Category', observed=False).agg({
  'MedHouseVal': ['mean', 'median'],
  'AveRooms': 'mean',
  'AveBedrms': 'mean'
}).round(2)

print("\n=== Statystyki według kategorii dochodów ===")
print(income_stats)

=== Liczba obserwacji według regionu ===
Region
NE    1549
NW    1490
SE     981
SW     980
Name: count, dtype: int64

=== Statystyki według regionu ===
       MedHouseVal              MedInc        Population          HouseAge
              mean median   std   mean median       mean      sum     mean
Region                                                                    
NE            2.51   2.33  1.35   4.19   3.53    2510.68  3889049    28.16
NW            2.46   2.19  1.34   4.09   3.39    2491.20  3711892    28.01
SE            1.89   1.66  1.35   4.13   3.57    2515.64  2467839    27.67
SW            1.88   1.61  1.32   4.08   3.58    2427.44  2378888    27.77

=== Statystyki według kategorii dochodów ===
                MedHouseVal        AveRooms AveBedrms
                       mean median     mean      mean
Income_Category                                      
Low                    0.92   0.88     5.57      1.12
Medium                 1.93   1.92     5.57      1.11
High  

In [45]:
# === FEATURE ENGINEERING ===
# 1. Agregacje regionalne jako cechy
df_housing['Avg_Price_Region'] = df_housing.groupby('Region')['MedHouseVal'].transform('mean')
df_housing['Avg_Income_Region'] = df_housing.groupby('Region')['MedInc'].transform('mean')

# 2. Odchylenie od średniej regionalnej
df_housing['Price_vs_Region'] = df_housing['MedHouseVal'] - df_housing['Avg_Price_Region']
df_housing['Income_vs_Region'] = df_housing['MedInc'] - df_housing['Avg_Income_Region']

# 3. Ratio features
df_housing['Rooms_per_Bedroom'] = df_housing['AveRooms'] / (df_housing['AveBedrms'] + 0.01)
df_housing['Population_per_House'] = df_housing['Population'] / (df_housing['AveOccup'] + 0.01)

# 4. Standaryzacja w grupach
for col in ['MedInc', 'HouseAge', 'AveRooms']:
  df_housing[f'{col}_Std_Region'] = df_housing.groupby('Region')[col].transform(
    lambda x: (x - x.mean()) / x.std()
  )

print("\n=== Dataset z feature engineering ===")
print(df_housing[['Region', 'MedHouseVal', 'Avg_Price_Region', 'Price_vs_Region',
                  'Rooms_per_Bedroom', 'MedInc_Std_Region']].head(10))


=== Dataset z feature engineering ===
  Region  MedHouseVal  Avg_Price_Region  Price_vs_Region  Rooms_per_Bedroom  \
0     NW       0.4077          2.455750        -2.048050           4.137893   
1     NE       4.0278          2.508611         1.519189           5.166382   
2     NE       3.2236          2.508611         0.714989           5.372420   
3     NW       3.0991          2.455750         0.643350           4.076433   
4     SW       2.2712          1.876426         0.394774           5.336925   
5     NW       2.0857          2.455750        -0.370050           6.264473   
6     NE       1.8484          2.508611        -0.660211           5.967175   
7     NE       1.7453          2.508611        -0.763311           4.490297   
8     SE       1.9495          1.890601         0.058899           4.772310   
9     NW       1.8094          2.455750        -0.646350           4.850484   

   MedInc_Std_Region  
0          -1.186429  
1           1.089427  
2           0.253526  

In [47]:
# Zamiast 4 wykresów matplotlib — 4 analizy tekstowe

print("=" * 60)
print("1. Średnia cena domu według regionu")
print("=" * 60)
avg_price_region = df_housing.groupby('Region')['MedHouseVal'].mean().sort_values()
print(avg_price_region.round(3))
max_v = avg_price_region.max()
print("\nTekstowe porównanie (█ ≈ 0.1 * max):")
for region, val in avg_price_region.items():
  bar = '█' * int(val / max_v * 30)
  print(f" {region:2s} |{bar} {val:.3f}")

print("\n" + "=" * 60)
print("2. Średnia cena według kategorii dochodów")
print("=" * 60)
avg_price_income = df_housing.groupby('Income_Category', observed=False)['MedHouseVal'].mean()
print(avg_price_income.round(3))
max_v = avg_price_income.max()
print("\nTekstowe porównanie (█ ≈ 0.033 * max):")
for cat, val in avg_price_income.items():
  bar = '█' * int(val / max_v * 30)
  print(f" {str(cat):10s} |{bar} {val:.3f}")

print("\n" + "=" * 60)
print("3. Rozkład ceny według wieku domu (kwartyle)")
print("=" * 60)
price_by_age = df_housing.groupby('Age_Category', observed=False)['MedHouseVal'].describe()[['min', '25%', '50%', '75%', 'max']].round(3)
print(price_by_age)

print("\n" + "=" * 60)
print("4. Zależność: średnia cena vs kategoria dochodów, w każdym regionie")
print("=" * 60)
cross = df_housing.pivot_table(
  index='Region',
  columns='Income_Category',
  values='MedHouseVal',
  aggfunc='mean',
  observed=False
).round(3)
print(cross)

1. Średnia cena domu według regionu
Region
SW    1.876
SE    1.891
NW    2.456
NE    2.509
Name: MedHouseVal, dtype: float64

Tekstowe porównanie (█ ≈ 0.1 * max):
 SW |██████████████████████ 1.876
 SE |██████████████████████ 1.891
 NW |█████████████████████████████ 2.456
 NE |██████████████████████████████ 2.509

2. Średnia cena według kategorii dochodów
Income_Category
Low          0.919
Medium       1.934
High         2.943
Very High    4.342
Name: MedHouseVal, dtype: float64

Tekstowe porównanie (█ ≈ 0.033 * max):
 Low        |██████ 0.919
 Medium     |█████████████ 1.934
 High       |████████████████████ 2.943
 Very High  |██████████████████████████████ 4.342

3. Rozkład ceny według wieku domu (kwartyle)
               min    25%    50%    75%  max
Age_Category                                
New           0.15  1.429  2.285  3.345  5.0
Modern        0.15  1.267  2.056  3.184  5.0
Old           0.15  1.170  1.996  3.084  5.0
Very Old      0.15  0.978  1.791  3.027  5.0

4. Zależnoś

In [52]:
# Zadanie 1. Podstawowe Grupowanie

import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Średni wiek według klasy i płci
tabela = df.groupby(["Pclass", "Sex"])["Age"].mean().round(2).unstack()
print(tabela)

print(tabela)

Sex     female   male
Pclass               
1        34.61  41.28
2        28.72  30.74
3        21.75  26.51
Sex     female   male
Pclass               
1        34.61  41.28
2        28.72  30.74
3        21.75  26.51


In [66]:
#Zadanie 2. Zliczanie Obserwacji
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Liczba pasażerów dla każdej kombinacji klasy i statusu przeżycia
liczba = df.groupby(["Pclass", "Survived"]).size()
print(liczba)

# Tabela pivot przez unstack()
tabela = liczba.unstack()
tabela.columns = ["Nie przezyl (0)", "Przezyl (1)"]
print("\n")
print(tabela)

#Tabela pivot przez df.pivot_table
tabela2 = df.pivot_table(
    index="Pclass",
    columns="Survived",
    values="PassengerId",
    aggfunc="count"
)
print("\n")
print(tabela2)

Pclass  Survived
1       0            80
        1           136
2       0            97
        1            87
3       0           372
        1           119
dtype: int64


        Nie przezyl (0)  Przezyl (1)
Pclass                              
1                    80          136
2                    97           87
3                   372          119


Survived    0    1
Pclass            
1          80  136
2          97   87
3         372  119


In [67]:
#Zadanie 3. Agregacja z .agg()
import numpy as np
import pandas as pd

np.random.seed(42)
sales = pd.DataFrame({
    'Region': np.random.choice(['A', 'B', 'C'], 100),
    'Sprzedaż': np.random.randint(50, 500, 100)
})

statystyki = (
    sales.groupby('Region')['Sprzedaż']
         .agg(['min', 'max', 'mean', 'std'])
         .round(2)
)
print(statystyki)

        min  max    mean     std
Region                          
A        64  495  285.94  128.35
B        54  495  255.89  137.73
C        77  498  262.35  126.50


In [68]:
# Zadanie 4. Transform
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Nowa kolumna: średnia opłata w klasie, przypisana każdemu wierszowi
df["Avg_Fare_Pclass"] = df.groupby("Pclass")["Fare"].transform("mean")

print(df[["Pclass", "Fare", "Avg_Fare_Pclass"]].head(10))

   Pclass     Fare  Avg_Fare_Pclass
0       3   7.2500        13.675550
1       1  71.2833        84.154687
2       3   7.9250        13.675550
3       1  53.1000        84.154687
4       3   8.0500        13.675550
5       3   8.4583        13.675550
6       1  51.8625        84.154687
7       3  21.0750        13.675550
8       3  11.1333        13.675550
9       2  30.0708        20.662183


In [74]:
# Zadanie 5. Binning Wieku
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Przedziały: (-1, 12], (12, 18], (18, 60], (60, 120]
przedzialy = [-1, 12, 18, 60, 120]
etykiety = ["Dziecko", "Nastolatek", "Dorosły", "Senior"]

df["Kategoria_wieku"] = pd.cut(df["Age"], bins=przedzialy, labels=etykiety)

# Survival rate (w %) i liczebność każdej kategorii
wyniki = (
    df.groupby("Kategoria_wieku", observed=True)["Survived"]
      .agg(Liczba="count", Przezylo="sum", Survival_rate="mean")
)
wyniki["Survival_rate"] = (wyniki["Survival_rate"] * 100).round(2)
print(wyniki)

print("\nLiczba pasazerow bez wieku (NaN)")
print(df["Kategoria_wieku"].isna().sum())

# Tekstowe słupki: 1 █ = 2%
print()
for kategoria, wiersz in wyniki.iterrows():
    slupek = "█" * int(wiersz["Survival_rate"] // 2)
    print(f"{kategoria:<11} | {slupek:<30} {wiersz['Survival_rate']:.2f}%")

                 Liczba  Przezylo  Survival_rate
Kategoria_wieku                                 
Dziecko              69        40          57.97
Nastolatek           70        30          42.86
Dorosły             553       215          38.88
Senior               22         5          22.73

Liczba pasazerow bez wieku (NaN)
177

Dziecko     | ████████████████████████████   57.97%
Nastolatek  | █████████████████████          42.86%
Dorosły     | ███████████████████            38.88%
Senior      | ███████████                    22.73%


In [75]:
# Zadanie 6. Feature Engineering: Family Size
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Rozmiar rodziny: rodzeństwo/małżonkowie + rodzice/dzieci + sam pasażer
df["Family_Size"] = df["SibSp"] + df["Parch"] + 1

# Survival rate (%) i liczebność dla każdego rozmiaru rodziny
wyniki = (
    df.groupby("Family_Size")["Survived"]
      .agg(Liczba="count", Przezylo="sum", Survival_rate="mean")
)
wyniki["Survival_rate"] = (wyniki["Survival_rate"] * 100).round(2)
print(wyniki)

# Rozmiar rodziny z najwyższym survival rate
najlepszy = wyniki["Survival_rate"].idxmax()
print(f"\nNajwyższy survival rate: Family_Size = {najlepszy} "
      f"({wyniki.loc[najlepszy, 'Survival_rate']:.2f}%, "
      f"n = {wyniki.loc[najlepszy, 'Liczba']})")

# Tekstowe słupki: 1 █ = 2%
print()
for rozmiar, wiersz in wyniki.iterrows():
    slupek = "█" * int(wiersz["Survival_rate"] // 2)
    print(f"Rodzina {rozmiar:>2} | {slupek:<50} {wiersz['Survival_rate']:.2f}% (n={wiersz['Liczba']})")

             Liczba  Przezylo  Survival_rate
Family_Size                                 
1               537       163          30.35
2               161        89          55.28
3               102        59          57.84
4                29        21          72.41
5                15         3          20.00
6                22         3          13.64
7                12         4          33.33
8                 6         0           0.00
11                7         0           0.00

Najwyższy survival rate: Family_Size = 4 (72.41%, n = 29)

Rodzina  1 | ███████████████                                    30.35% (n=537.0)
Rodzina  2 | ███████████████████████████                        55.28% (n=161.0)
Rodzina  3 | ████████████████████████████                       57.84% (n=102.0)
Rodzina  4 | ████████████████████████████████████               72.41% (n=29.0)
Rodzina  5 | ██████████                                         20.00% (n=15.0)
Rodzina  6 | ██████                       

In [77]:
# Zadanie 7. Grupowanie i Sortowanie
df = df_housing.copy()

# Siatka 4x4: 4 przedziały szerokości x 4 przedziały długości geograficznej
df["lat_bin"] = pd.cut(df["Latitude"], 4)
df["lon_bin"] = pd.cut(df["Longitude"], 4)
df["region"] = df["lat_bin"].astype(str) + "|" + df["lon_bin"].astype(str)

# Średnia cena według regionu, malejąco, top 5
top5 = (
    df.groupby("region")["MedHouseVal"]
      .agg(Liczba="count", Srednia_cena="mean")
      .round(2)
      .sort_values("Srednia_cena", ascending=False)
      .head(5)
)
print(top5)

                                      Liczba  Srednia_cena
region                                                    
(39.25, 41.498]|(-118.999, -116.749]     354          2.76
(39.25, 41.498]|(-121.248, -118.999]     298          2.72
(39.25, 41.498]|(-116.749, -114.5]       302          2.71
(39.25, 41.498]|(-123.506, -121.248]     287          2.67
(37.001, 39.25]|(-116.749, -114.5]       307          2.38


In [79]:
#Zadanie 8. Wypełnianie Braków Grupowo
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# 1. Braki w Age i statystyki PRZED
print("Braki przed:", df["Age"].isna().sum())
przed = df["Age"].describe().round(2)

# Mediany w klasach (do wglądu)
print(df.groupby("Pclass")["Age"].median())

# 2. Wypełnienie medianą grupową według Pclass
df["Age"] = df.groupby("Pclass")["Age"].transform(lambda x: x.fillna(x.median()))

# 3. Sprawdzenie PO
print("\nBraki po:", df["Age"].isna().sum())
po = df["Age"].describe().round(2)

# 4. Porównanie przed/po
print(pd.DataFrame({"przed": przed, "po": po}))

Braki przed: 177
Pclass
1    37.0
2    29.0
3    24.0
Name: Age, dtype: float64

Braki po: 0
        przed      po
count  714.00  891.00
mean    29.70   29.07
std     14.53   13.24
min      0.42    0.42
25%     20.12   22.00
50%     28.00   26.00
75%     38.00   37.00
max     80.00   80.00
